# AI-Augmented Investment Evaluation Pipeline
### IB9LQ0 — Generative AI and AI Applications | Group Assignment

**GDPval task:** `b78fd844-db76-448e-a783-5e9877cb74c2` — Senior Finance Manager / Investment Evaluation </br>
**Primary reference case:** Tiny-Rod Hit Inc.</br>
**Generalisation evidence:** four synthetic reference documents covering different sectors, missing inputs, weak-options scenarios, and multi-project capital allocation.

## Overview
This notebook implements a retrieval-augmented generation (RAG) pipeline that produces a Board of Directors investment evaluation report from a reference document. The pipeline is designed for traceability: every factual claim in the final report can be mapped back to a specific chunk of the source material, and every generation step is logged with prompts, model versions, token counts and cost.

## How to run
1. Run cells top-to-bottom: **Runtime -> Run all**
2. Paste your Anthropic API key when prompted (Cell B2)
3. Update config (for synthetic reference run in Cell B4)
4. Upload reference file (Cell B5)
5. Outputs appear in `/content/outputs/run_<id>/`


## Pipeline architecture

The pipeline runs as an eight-stage two-pass design. The recommendation is locked in Stage 2 before any prose is generated in Stages 3–5, which eliminates cross-section contradictions where different sections would otherwise recommend different projects.

| Stage | Purpose |
|---|---|
| 0 — Input adapter | Read the PDF/Word/text reference and normalise its text |
| 1 — Ingest & index | Chunk the text, embed with a hybrid BM25 + FAISS retriever, auto-extract canonical facts |
| 2 — Select recommendation | Lock in one recommended project before any section is written |
| 3 — Plan | Build section-specific retrieval queries from the extracted facts and the locked recommendation |
| 4 — Retrieve | Fetch the top-k relevant chunks per section using the hybrid index |
| 5 — Generate | Generate all nine sections conditioned on the locked recommendation |
| 6 — Verify | Three-layer verification: rubric-aligned regex checks, citation verifier, rubric critic |
| 7 — Render | Produce the Word/PDF report, confidence dashboard, evaluation report, and audit log |

## Output package

| File | What it is |
|---|---|
| `board_report.pdf` / `.docx` | The Board of Directors deliverable |
| `dashboard.html` | Confidence dashboard with rubric coverage and section-level scores |
| `evaluation_report.md` | Human-readable record of every stage including consistency verification |
| `audit_log.json` | Full provenance: prompts, model, scores, cost, locked decision |
| `verification_results.json` | All Layer A / B / C check results |
| `baseline_comparison.json` | Pipeline vs raw Claude comparison (B12) |

---

## B1 — Install dependencies

Installs the Python packages required by the pipeline, pinned to exact versions for reproducibility. The `rank-bm25` package powers the keyword side of the hybrid retriever in Stage 1, and `python-docx` is used by Stage 7 to build the final Word report including the programmatic NPV/IRR comparison table. LibreOffice is installed via `apt` so that the Word output can be converted to PDF inside the Colab runtime.

In [ ]:
!pip install -q \
    anthropic==0.40.0 \
    sentence-transformers==2.7.0 \
    faiss-cpu==1.8.0 \
    pypdf==4.3.1 \
    python-docx==1.1.2 \
    numpy==1.26.4 \
    rank-bm25==0.2.2

# Colab pre-installs numpy 2.x which conflicts with sentence-transformers.
# After pip installs 1.26.4 we must restart so Python loads the new version.
import numpy as np
if np.__version__.startswith("2"):
    print("numpy 2.x detected — restarting runtime...")
    import os; os.kill(os.getpid(), 9)
else:
    print(f"numpy {np.__version__} — OK")

!which soffice || apt-get install -y libreoffice -q
print("Ready")


## B2 — Set API key

Reads the Anthropic API key from Colab's secrets vault if available, otherwise prompts for it via `getpass` so the key is never displayed. The key is set on `os.environ['ANTHROPIC_API_KEY']` for the rest of the notebook.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('✅ API key loaded from Colab Secrets')
except Exception:
    import getpass
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')
    print('✅ API key set')


## B3 — Create folders

Creates the folder structure under `/content/`:
- `data/` — reference documents are placed here after upload
- `outputs/` — each pipeline run saves its full output package as `run_<id>/`, so multiple runs do not overwrite each other and a complete audit trail is preserved.

In [ ]:
from pathlib import Path
OUTPUT_DIR = Path('/content/outputs')
DATA_DIR = Path('/content/data')
for d in [OUTPUT_DIR, DATA_DIR]: d.mkdir(parents=True, exist_ok=True)
print('OK')


## B4 — Settings

Central configuration for the pipeline. All tunable parameters live here so behaviour can be adjusted without editing logic elsewhere.

**Model and embedding choices.** `LLM_MODEL` is set to `claude-haiku-4-5-20251001` for cost-efficient iteration; swap to a Sonnet model for a final production run. The embedding model `all-MiniLM-L6-v2` is small, fast and runs locally with no external API call.

**Retrieval parameters.** Chunks are 1,400 characters with 200-character overlap to balance retrieval focus against contextual continuity at chunk boundaries. The top `TOP_K` chunks per query are passed to the generator; this is enough to cover a section's grounding without bloating the context window. `CONFIDENCE_THRESHOLD = 0.70` gates which sections trigger an automatic revision cycle in Stage 6 and which sections are escalated for human review.

**Task-level constraints.** `TASK_CONFIG` provides per-task values for fields the reference document is silent on. Two configurations are provided:

- The first (active) block contains values supplied by the GDPval task brief for the Tiny-Rod case (company name, WACC, available cash, analysis date). The Tiny-Rod reference PDF does not state these — they are part of the task context surrounding the document.
- The second (commented-out) block sets all five fields to `None` and is used when running the four synthetic reference documents, which state these values within the document text. With the empty block active, auto-extraction takes precedence and `TASK_CONFIG` provides no overrides.

To switch between runs, swap which block is active. The override loop in Stage 1 is a fallback (only fills gaps), so the design separates document content from external task context cleanly.

**Section templates.** `SECTIONS_TEMPLATE` defines the nine-section structure of the Board report. Each section specifies its retrieval queries and writing instructions. The `{REC}` placeholder is substituted at runtime with the recommended project name selected in Stage 2, anchoring every section to the same decision.

In [ ]:
from datetime import datetime

# TASK_CONFIG provides values for fields the reference document is silent on.
# Two configurations are defined below: the active one is for the Tiny-Rod case
# (where the reference document does not state these values, so the GDPval task context supplies them)
# The commented-out empty block is used when running on the synthetic documents that state these values within the text.
# The override loop in Stage 1 only fills gaps, so extracted values take precedence when present.

TASK_CONFIG = {
    "company_name":           "Tiny-Rod Hit Inc.",   # GDPval task context
    "wacc":                   "9%",                  # GDPval task context
    "available_cash":         "$100 million",        # GDPval task context
    "analysis_prepared":      "January 2025",        # GDPval task context
    "analysis_current_as_of": "May 2025",            # GDPval task context
}


'''TASK_CONFIG = {
    "company_name":           None,
    "wacc":                   None,
    "available_cash":         None,
    "analysis_prepared":      None,
    "analysis_current_as_of": None,
}'''

LLM_MODEL       = "claude-haiku-4-5-20251001"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
COST_INPUT_PER_MTOK  = 0.80
COST_OUTPUT_PER_MTOK = 4.00

CHUNK_SIZE_CHARS     = 1400
CHUNK_OVERLAP        = 200
TOP_K                = 3
CONFIDENCE_THRESHOLD = 0.70

SECTION_TITLES = {
    "executive_summary":       "Executive Summary",
    "introduction":            "1. Introduction",
    "project_overview":        "2. Project Overview",
    "financial_analysis":      "3. Financial Analysis",
    "recommendation":          "4. Recommendation",
    "risk_mitigation":         "5. Risk Mitigation",
    "capital_allocation":      "6. Capital Allocation",
    "organisational_capacity": "7. Organisational Capacity",
    "conclusion":              "8. Conclusion",
}

# Section templates. The {REC} placeholder is substituted at runtime with the project selected by Stage 2
# every section is anchored to the same recommendation and cannot drift.
SECTIONS_TEMPLATE = [
    {"name": "executive_summary",
     "queries": ["investment overview projects financial position", "company cash WACC"],
     "instruction": (
         "Write the EXECUTIVE SUMMARY (~250 words). "
         "OPEN WITH EXACTLY: 'RECOMMENDATION: Proceed with {REC} as the priority investment.' "
         "Then summarise: (1) projects evaluated; (2) why {REC} was selected over alternatives; "
         "(3) capital allocation approach if both pursued. "
         "Do NOT contradict the {REC} recommendation anywhere in this section."
     ),
     "max_tokens": 700},

    {"name": "introduction",
     "queries": ["company context strategic situation fiscal year"],
     "instruction": (
         "Write the INTRODUCTION (~200 words). "
         # Timing context is mandatory in the Introduction.
         "OPEN WITH: 'This analysis is prepared in {PREPARED}, with information current as of {CURRENT}.' "
         "If PREPARED is not stated, omit it and open with: 'This analysis reflects information current as of {CURRENT}.' "
         "Then describe: company profile, available cash, WACC as hurdle rate, "
         "projects under evaluation, and analysis scope."
     ),
     "max_tokens": 600},

    {"name": "project_overview",
     "queries": [],
     "instruction": (
         "Write PROJECT OVERVIEW (~350 words). "
         "Each project in its own subsection with these four headings: "
         "Investment & Scope; Revenue Trajectory; Key Risks; Strategic Fit. "
         "Equal depth for all projects — do not pre-empt the recommendation here."
     ),
     "max_tokens": 1200},

    {"name": "financial_analysis",
     "queries": [],
     "instruction": (
         "Write FINANCIAL ANALYSIS (~500 words). "
         "For EACH project provide: "
         "(1) Directional NPV at 9% WACC (positive/negative/near-breakeven — NO point estimates); "
         "(2) Directional IRR vs 9% WACC (above/below/near WACC); "
         # Upside/downside scenarios are mandatory to satisfy the rubric exhibit requirement.
         "(3) UPSIDE SCENARIO: explicit description of what occurs if growth assumptions hold, "
         "tied to the specific cash-flow drivers from the reference (growth rate, duration, capex); "
         "(4) DOWNSIDE SCENARIO: explicit description of what occurs if key risks materialise, "
         "tied to specific risk drivers (failure probability, geopolitical, currency, regulatory). "
         "End the section with a clear comparative statement naming which project has the higher "
         "directional NPV at 9% and which has the higher directional IRR. "
         "Build toward the {REC} recommendation but state it formally only in the Recommendation section."
     ),
     "max_tokens": 1500},

    {"name": "recommendation",
     "queries": [],
     "instruction": (
         "Write the RECOMMENDATION section (~300 words). "
         "STATE EXPLICITLY in the opening sentence: 'The recommendation is to proceed with {REC}.' "
         "Justify {REC} using: "
         "(a) quantitative reasons consistent with the directional NPV/IRR analysis at 9% WACC; "
         "(b) at least two qualitative reasons (strategic fit, operational capability, "
         "execution feasibility); "
         "(c) explicit comparison: why {REC} is preferred over the alternative(s). "
         "DO NOT recommend any project other than {REC}. Do not waver or hedge."
     ),
     "max_tokens": 800},

    {"name": "risk_mitigation",
     "queries": [],
     # Risks are scoped to the recommended project only to avoid cross-project mixing
     "instruction": (
         "Write RISK MITIGATION (~500 words). "
         "Identify EXACTLY 3 risks SPECIFIC TO {REC} ONLY — do NOT include risks from any other project. "
         "At least 1 must be a financial risk and at least 1 must be an operational risk. "
         "Label each as 5.1, 5.2, 5.3. "
         "For each risk provide three sub-headings: Description; Mitigation Strategy; Contingency Plan. "
         "All three risks must clearly tie back to {REC} factual drivers from the reference."
     ),
     "max_tokens": 1500},

    {"name": "capital_allocation",
     "queries": ["capital allocation strategic balance diversification value"],
     "instruction": (
         "Write CAPITAL ALLOCATION (~350 words). "
         "Despite the {REC} priority, the Board has asked: 'If both projects are viable, how would $100M be allocated?' "
         "Provide concrete dollar amounts for EACH project plus reserve, summing to no more than $100M. "
         "Address: (1) long-term value creation beyond project-specific returns; "
         "(2) diversification benefits / concentration trade-offs; "
         "(3) strategic alignment with company objectives — cite at least one reference-grounded factor per project; "
         "(4) the company's strong balance sheet and healthy debt-to-equity as enabling factors; "
         "(5) why {REC} receives priority weighting if different from equal allocation."
     ),
     "max_tokens": 900},

    {"name": "organisational_capacity",
     "queries": ["organisational capability strengths gaps talent"],
     "instruction": (
         "Write ORGANISATIONAL CAPACITY (~250 words). "
         "Strengths supporting {REC}; capability gaps for the alternative project(s); "
         "bandwidth constraints if both pursued; talent risk; learning opportunity."
     ),
     "max_tokens": 700},

    {"name": "conclusion",
     "queries": ["recommendation next steps"],
     "instruction": (
         "Write CONCLUSION (~150 words). "
         "OPEN WITH: 'In conclusion, {REC} is the recommended priority for capital deployment.' "
         "Reaffirm the {REC} priority, the allocation approach, and provide 3-4 specific next steps. "
         "Do NOT recommend any project other than {REC}."
     ),
     "max_tokens": 500},
]

SECTIONS = SECTIONS_TEMPLATE
print(f"Config loaded: {LLM_MODEL}, {len(SECTIONS)} sections")


Config loaded: claude-haiku-4-5-20251001, 9 sections


## B5 — Upload reference document

Upload the reference PDF, Word document or plain text file. Stage 0 (Cell B7) auto-detects the format and extracts the text accordingly. The uploaded file is moved into `/content/data/`, with any whitespace or special characters in the filename replaced by underscores so the audit-log source path stays clean. Its final path is stored in `REFERENCE_PATH`, which the rest of the pipeline reads from.

**Supported formats:** `.pdf`, `.docx`, `.txt`

In [ ]:
from google.colab import files as colab_files
import shutil, re

print("Upload your reference document (PDF, .txt, or .docx)")

uploaded = colab_files.upload()

# Move uploaded file to data folder and set as active reference.
# Whitespace and special characters in the filename are replaced with underscores
# so the audit-log source_path does not contain URL-encoded characters like %20.
for fname in uploaded:
    clean = re.sub(r"[^A-Za-z0-9._-]+", "_", fname).strip("_")
    dest = DATA_DIR / clean
    shutil.move(fname, dest)
    REFERENCE_PATH = dest
    print(f"✓ Uploaded: {dest}")

print(f"\nActive reference: {REFERENCE_PATH.name}")

Upload your reference document (PDF, .txt, or .docx)


Saving Tiny%20Rod%20Hit%20Inc%20Reference.pdf to Tiny%20Rod%20Hit%20Inc%20Reference.pdf
✓ Uploaded: /content/data/Tiny_20Rod_20Hit_20Inc_20Reference.pdf

Active reference: Tiny_20Rod_20Hit_20Inc_20Reference.pdf


## B6 — LLM wrapper

A single function (`llm_call`) handles every interaction with the language model. All pipeline stages route through this wrapper, which gives the architecture three useful properties:

- **Model-agnostic.** Switching providers requires changing one function.
- **Cost-tracked.** Every call logs token usage and dollar cost into `_token_log`, which feeds the final `audit_log.json`.
- **Resilient.** Up to three retries with exponential backoff so the pipeline survives transient API failures.

In [ ]:
import time
from anthropic import Anthropic, APIError, RateLimitError

_client = None
_token_log = []

def get_client():
    global _client
    if _client is None:
        _client = Anthropic()
    return _client

def llm_call(system, user, max_tokens=1200, temperature=0.3, purpose="gen"):
    client = get_client()
    for attempt in range(3):
        try:
            resp = client.messages.create(
                model=LLM_MODEL, max_tokens=max_tokens, temperature=temperature,
                system=system, messages=[{"role": "user", "content": user}],
            )
            tin  = resp.usage.input_tokens
            tout = resp.usage.output_tokens
            cost = (tin * COST_INPUT_PER_MTOK + tout * COST_OUTPUT_PER_MTOK) / 1_000_000
            _token_log.append({
                "purpose": purpose,
                "system_prompt_preview": system[:800],
                "user_prompt_preview": user[:1200],
                "tokens_in": tin,
                "tokens_out": tout,
                "cost_usd": cost
            })
            return {"text": resp.content[0].text, "tokens_in": tin,
                    "tokens_out": tout, "cost_usd": cost}
        except Exception as e:
            wait = 15 * (attempt + 1)
            print(f"  [retry {attempt+1}/3] {type(e).__name__}: {str(e)[:150]} — waiting {wait}s")
            time.sleep(wait)
    raise RuntimeError("LLM call failed after 3 retries")

def total_cost(): return sum(c["cost_usd"] for c in _token_log)
def reset_log():
    global _token_log; _token_log = []

print("LLM wrapper ready")


LLM wrapper ready


## B7 — Stage 0 + Stage 1: Ingest and index

**Stage 0** reads the document, normalises whitespace, repairs broken line breaks from PDF extraction, and runs a health check. If less than 100 characters of text are extracted, the stage fails loudly with a clear error rather than silently producing garbage downstream — this guards against scanned PDFs being misinterpreted.

**Stage 1** does three things:
1. Chunks the cleaned text using a sliding window with sentence-boundary breaks
2. Builds a `HybridRetriever` index over the chunks
3. Calls the LLM (at zero temperature) to auto-extract canonical facts: company name, WACC, available cash, analysis timing, and per-project investment, growth and failure-probability data.

**Why a hybrid retriever.** Dense embeddings from `all-MiniLM-L6-v2` capture semantic similarity well but can miss exact financial terms such as `$40 million` or `30% failure probability` because the embedding model encodes meaning rather than literal strings. BM25 is a keyword-based scorer that catches those exact matches. The two methods are combined via Reciprocal Rank Fusion (`score = 1 / (60 + rank)` summed across methods), which improves retrieval robustness on finance documents compared to either method alone.

**Constraint layer.** The extracted facts become the anti-hallucination guardrails injected into every downstream generation prompt. Where a value is genuinely not stated in the reference, the system prompt explicitly marks it "NOT STATED — do not invent," forcing abstention rather than fabrication. Where `TASK_CONFIG` (from B4) provides a per-task fallback, that value is used instead.

In [ ]:
import re, json, hashlib, unicodedata, numpy as np, faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi  # keyword scorer for the hybrid retriever

def normalise(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"-\s*\n\s*", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def stage0_read(path):
    path = Path(path)
    if path.suffix.lower() == ".pdf":
        reader = PdfReader(str(path))
        text = " ".join(p.extract_text() or "" for p in reader.pages)
    elif path.suffix.lower() == ".docx":
        from docx import Document as DocxReader
        doc = DocxReader(str(path))
        text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
    else:
        text = path.read_text(encoding="utf-8")

    text = normalise(text)
    if len(text) < 100:
        raise RuntimeError("No text extracted. PDF may be scanned.")

    src_hash = hashlib.sha256(path.read_bytes()).hexdigest()[:16]
    print(f"  Extracted {len(text):,} chars from {path.name}")
    return {"text": text, "source_path": str(path), "source_hash": src_hash}

_embedder = None
def get_embedder():
    global _embedder
    if _embedder is None:
        _embedder = SentenceTransformer(EMBEDDING_MODEL)
    return _embedder

def chunk_text(text):
    chunks = []
    start = 0
    cid = 0
    while start < len(text):
        end = min(start + CHUNK_SIZE_CHARS, len(text))
        if end < len(text):
            lp = text.rfind(".", start, end)
            if lp > start + CHUNK_SIZE_CHARS // 2:
                end = lp + 1
        chunks.append({"chunk_id": cid, "text": text[start:end].strip(), "source": "ref"})
        cid += 1
        start = end - CHUNK_OVERLAP if end < len(text) else end
    return chunks

FACT_SYS = """Extract canonical facts from an investment reference document.
Look carefully for: company name, WACC or cost of capital percentage,
total available cash, fiscal year or analysis date if stated, and project details.

Return strict JSON:
{"company_name": str|null, "wacc": str|null, "available_cash": str|null,
 "analysis_prepared": str|null, "analysis_current_as_of": str|null,
 "projects": [{"name": str, "investment": str, "growth_profile": str,
               "failure_probability": str|null, "key_risks": [str]}]}
Use null only if genuinely not stated anywhere. Never invent."""

def extract_facts(text):
    r = llm_call(FACT_SYS, f"REFERENCE:\n---\n{text}\n---\nReturn strict JSON.",
                 max_tokens=1000, temperature=0.0, purpose="facts")
    raw = re.sub(r"^```(?:json)?|```$", "", r["text"].strip(), flags=re.MULTILINE).strip()
    try: return json.loads(raw)
    except: print("  [warn] fact extraction: invalid JSON"); return {}


# HybridRetriever combines dense (FAISS) and keyword (BM25) retrieval via Reciprocal Rank Fusion.
# Dense retrieval captures semantic similarity;
# BM25 catches exact financial terms (e.g. "$40 million", "30% failure") that the embedding model can miss.
# Each method ranks chunks independently;
# RRF combines them with score = 1 / (60 + rank), summed across methods.
# Top-k by combined score are returned.
class HybridRetriever:
    def __init__(self, chunks):
        self.chunks = chunks
        self.chunk_map = {c["chunk_id"]: c for c in chunks}

        # --- Dense index (FAISS) ---
        m = get_embedder()
        emb = np.asarray(
            m.encode([c["text"] for c in chunks],
                     normalize_embeddings=True, show_progress_bar=False),
            dtype="float32"
        )
        self.faiss_idx = faiss.IndexFlatIP(emb.shape[1])
        self.faiss_idx.add(emb)

        # --- Sparse index (BM25) ---
        tokenised = [c["text"].lower().split() for c in chunks]
        self.bm25 = BM25Okapi(tokenised)

    def retrieve(self, query, k=TOP_K):
        fetch_k = min(k * 3, len(self.chunks))  # fetch more, then fuse

        # Dense retrieval
        m = get_embedder()
        q_emb = np.asarray(
            m.encode([query], normalize_embeddings=True), dtype="float32"
        )
        _, faiss_idxs = self.faiss_idx.search(q_emb, fetch_k)
        dense_ranks = {self.chunks[i]["chunk_id"]: rank
                       for rank, i in enumerate(faiss_idxs[0])}

        # Sparse retrieval (BM25)
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = sorted(range(len(self.chunks)),
                             key=lambda i: -bm25_scores[i])[:fetch_k]
        bm25_ranks  = {self.chunks[i]["chunk_id"]: rank
                       for rank, i in enumerate(bm25_ranked)}

        # Reciprocal Rank Fusion
        all_ids = set(dense_ranks) | set(bm25_ranks)
        rrf_scores = {}
        for cid in all_ids:
            score = 0.0
            if cid in dense_ranks: score += 1 / (60 + dense_ranks[cid])
            if cid in bm25_ranks:  score += 1 / (60 + bm25_ranks[cid])
            rrf_scores[cid] = score

        top_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:k]
        return [self.chunk_map[cid] for cid in top_ids if cid in self.chunk_map]
# ══════════════════════════════════════════════════════════════════════


def stage1_ingest(s0, run_dir):
    chunks = chunk_text(s0["text"])
    print(f"  {len(chunks)} chunks")

    # [CHANGE 4] Use HybridRetriever instead of dense-only Retriever
    retriever = HybridRetriever(chunks)
    print("  Hybrid BM25+FAISS index built")

    facts = extract_facts(s0["text"])

    # ══════════════════════════════════════════════════════════════════
    # Apply TASK_CONFIG overrides instead of silent hardcoding
    #
    # ORIGINAL CODE (removed):
    #   if not facts.get("company_name"): facts["company_name"] = "Tiny-Rod Hit Inc."
    #   if not facts.get("wacc"):         facts["wacc"] = "9%"
    #   if not facts.get("available_cash"): facts["available_cash"] = "$100 million"
    #
    # PROBLEM: Those three lines silently injected Tiny-Rod's values into
    # any document, making the pipeline produce wrong reports for other tasks.
    #
    # FIX: Apply TASK_CONFIG values (set in B4) which override extraction,
    # then warn clearly if a critical fact is still missing.
    # ══════════════════════════════════════════════════════════════════
    for key in ("company_name", "wacc", "available_cash"):
      override = TASK_CONFIG.get(key)
      if override and not facts.get(key):   # only fill gaps, never overwrite
          facts[key] = override
          print(f"  [config fallback] {key} = {override}")

    missing = [k for k in ("wacc", "available_cash")
               if not facts.get(k)]
    if missing:
        print(f"  ⚠ WARNING: Could not extract {missing} from document.")
        print(f"    Set them in TASK_CONFIG in B4, or the report may be incomplete.")
    # ══════════════════════════════════════════════════════════════════

    print(f"  Facts: company={facts.get('company_name')} "
          f"wacc={facts.get('wacc')} cash={facts.get('available_cash')}")
    print(f"  Projects found: {[p.get('name') for p in facts.get('projects', [])]}")

    (run_dir / "facts.json").write_text(json.dumps(facts, indent=2))
    (run_dir / "chunks.json").write_text(json.dumps(chunks, indent=2))
    return {"chunks": chunks, "facts": facts, "retriever": retriever}

print("Stages 0-1 defined")


Stages 0-1 defined


## B8 — Stage 2: Select recommendation + Stages 3-5: Plan, retrieve, generate

A two-pass architecture is used to eliminate the cross-section contradictions that arise when each section is generated independently from the same retrieval pool.

**Pass 1 — `stage2_select_recommendation()`** runs before any section is written. It retrieves project-comparison chunks, calls the LLM with a strict JSON schema and explicit selection criteria (directional NPV, IRR, strategic fit, risk profile, organisational capability), parses the result, and stores `recommended_project` along with its rationale in the canonical facts dictionary.

**Pass 2 — `stage_345_generate()`** then writes all nine sections with the locked decision injected into every system prompt. The `{REC}` placeholder in each section's instruction is substituted with the selected project name at runtime by `build_dynamic_sections()` (Stage 3), the top-k chunks per section are retrieved using the hybrid index (Stage 4), and the LLM generates the prose (Stage 5). Because every prompt now references the same recommendation, the Executive Summary, Recommendation and Conclusion cannot drift apart.

**Prompt engineering principles** applied at the system-prompt level for every section:
1. **Locked-in facts.** Extracted values are stated as the only figures the model may use. Missing values are flagged "NOT STATED — do not invent."
2. **Directional language only.** Specific NPV and IRR point estimates are banned because the reference contains no cash-flow projections; the model must use phrases such as "directionally positive at the 9% WACC."
3. **Mandatory citations.** Every factual claim must terminate in a `[chunk_N]` tag referencing its source chunk.

In [ ]:
import copy

# build_dynamic_sections populates per-section retrieval queries from the
# extracted project names and substitutes the {REC} placeholder in each
# section instruction with the project locked in by Stage 2.
def build_dynamic_sections(facts):
    projects = facts.get("projects", [])
    proj_names = [p.get("name", f"Project {i+1}") for i, p in enumerate(projects)]
    rec = facts.get("recommended_project", proj_names[0] if proj_names else "the recommended project")

    sections = copy.deepcopy(SECTIONS_TEMPLATE)

    for sec in sections:
        # Fill queries dynamically
        if sec["name"] == "project_overview":
            sec["queries"] = [f"{n} investment growth risks strategic fit" for n in proj_names] \
                             or ["project investment growth risks"]
        elif sec["name"] == "financial_analysis":
            sec["queries"] = [f"{n} NPV IRR financial implications" for n in proj_names] \
                             + ["strengths weaknesses comparison WACC"]
        elif sec["name"] == "recommendation":
            sec["queries"] = [f"compare {' and '.join(proj_names)} strengths weaknesses"] \
                             + [f"{n} strategic fit operational risk" for n in proj_names]
        elif sec["name"] == "risk_mitigation":
            sec["queries"] = [f"{rec} financial risk",
                              f"{rec} operational risk",
                              f"{rec} strategic risk"]

        # Substitute {REC} placeholder with locked recommendation
        sec["instruction"] = sec["instruction"].replace("{REC}", rec)

    return sections


# Stage 2 selects ONE recommended project before any section is generated,
# so every downstream section is conditioned on the same anchor and cannot
# drift to a different recommendation.
SELECT_SYS = '''You are a Senior Finance Manager preparing a Board of Directors report.

You must select ONE project to recommend as the priority investment, based ONLY
on the reference material provided.

Return STRICT JSON only (no markdown, no commentary):
{
  "recommended_project": "<exact project name from facts>",
  "rationale_summary": "<2-3 sentence Board-facing justification>",
  "quantitative_drivers": ["driver 1", "driver 2"],
  "qualitative_drivers": ["driver 1", "driver 2"]
}

Selection criteria (in order):
1. Directional NPV at the company WACC
2. Directional IRR vs WACC
3. Strategic fit with stated company objectives
4. Risk profile (favouring quantifiable risks over execution uncertainty)
5. Organisational capability fit

Use directional language only. Do NOT invent exact NPV/IRR figures.'''


def stage2_select_recommendation(retriever, facts):
    """
    Pass 1: Lock in the recommended project BEFORE generating any sections.
    Returns the decision dict and updates facts in place.
    """
    # Retrieve broad comparison context
    queries = [
        "compare projects strengths weaknesses",
        "investment growth risks failure",
        "strategic fit competitive advantage",
        "WACC hurdle rate financial return",
    ]
    retrieved, seen = [], set()
    for q in queries:
        for c in retriever.retrieve(q, k=3):
            if c["chunk_id"] not in seen:
                retrieved.append(c)
                seen.add(c["chunk_id"])
    context = "\n\n".join(f"[chunk_{c['chunk_id']}]: {c['text']}" for c in retrieved)

    proj_list = [p.get("name", "?") for p in facts.get("projects", [])]
    proj_summary = "\n".join(
        f"- {p.get('name')}: invest {p.get('investment')}, "
        f"growth {p.get('growth_profile')}, "
        f"failure prob {p.get('failure_probability') or 'not stated'}"
        for p in facts.get("projects", [])
    )

    user_msg = (
        f"COMPANY: {facts.get('company_name', 'the company')}\n"
        f"WACC: {facts.get('wacc', 'not stated')}\n"
        f"AVAILABLE CASH: {facts.get('available_cash', 'not stated')}\n\n"
        f"PROJECTS:\n{proj_summary}\n\n"
        f"REFERENCE CONTEXT:\n{context}\n\n"
        f"Select ONE of: {', '.join(proj_list)}. Return JSON only."
    )

    r = llm_call(SELECT_SYS, user_msg, max_tokens=600, temperature=0.0, purpose="select_rec")
    raw = re.sub(r"^```(?:json)?|```$", "", r["text"].strip(), flags=re.MULTILINE).strip()

    try:
        decision = json.loads(raw)
        # Validate the selected project is one of the real projects
        if decision.get("recommended_project") not in proj_list:
            print(f"  [warn] LLM picked '{decision.get('recommended_project')}', not in {proj_list}")
            decision["recommended_project"] = proj_list[0] if proj_list else "Project A"
    except Exception as e:
        print(f"  [warn] selection JSON parse failed: {e}")
        decision = {
            "recommended_project": proj_list[0] if proj_list else "Project A",
            "rationale_summary": "Manual review required — automated selection failed",
            "quantitative_drivers": [],
            "qualitative_drivers": [],
        }

    # Update facts in place — this becomes a canonical fact for all downstream sections
    facts["recommended_project"] = decision["recommended_project"]
    facts["recommendation_rationale"] = decision.get("rationale_summary", "")
    facts["quantitative_drivers"] = decision.get("quantitative_drivers", [])
    facts["qualitative_drivers"] = decision.get("qualitative_drivers", [])

    return decision
# ══════════════════════════════════════════════════════════════════════


def build_system_prompt(facts):
    company = facts.get("company_name") or "the company"
    wacc    = facts.get("wacc")
    cash    = facts.get("available_cash")
    projs   = facts.get("projects", [])

    lines = [f"Company: {company}"]
    lines.append(f"WACC: exactly {wacc}" if wacc
                 else "WACC: NOT STATED. Do NOT invent. Say so explicitly if asked.")
    lines.append(f"Available cash: exactly {cash}" if cash
                 else "Available cash: NOT STATED. Do NOT invent.")

    # Inject timing context so the Introduction can reference it.
    # analysis_prepared: extracted from reference or marked as not stated
    # analysis_current_as_of: auto-generated from system clock if not in TASK_CONFIG
    prepared = facts.get("analysis_prepared") or TASK_CONFIG.get("analysis_prepared") or "not stated"
    current  = facts.get("analysis_current_as_of") or TASK_CONFIG.get("analysis_current_as_of") or datetime.now().strftime("%B %Y")

    # Write the resolved timing values back to facts so audit_log.json and
    # facts.json accurately reflect what was used downstream.
    facts["analysis_prepared"] = prepared if prepared != "not stated" else None
    facts["analysis_current_as_of"] = current

    if prepared == "not stated":
        lines.append(f"Timing: current as of {current}. Preparation date not stated — do not fabricate.")
    else:
        lines.append(f"Timing: analysis prepared {prepared}, current as of {current}.")

    if projs:
        lines.append("Projects:")
        for p in projs:
            ln = f"  - {p.get('name','?')}: investment {p.get('investment','not stated')}"
            if p.get("failure_probability"):
                ln += f", failure probability {p['failure_probability']}"
            lines.append(ln)

    # The locked decision is injected into every section prompt below.
    rec = facts.get("recommended_project")
    if rec:
        lines.append("")
        lines.append("=== LOCKED RECOMMENDATION (binding for all sections) ===")
        lines.append(f"Recommended project: {rec}")
        if facts.get("recommendation_rationale"):
            lines.append(f"Rationale: {facts['recommendation_rationale']}")
        lines.append("Every section must be consistent with this recommendation.")
        lines.append("Do NOT recommend any other project anywhere in the report.")
        lines.append("===")

    return f'''You are a Senior Finance Manager writing a Board of Directors report section.

CANONICAL FACTS (only these figures may be stated):
{chr(10).join(lines)}

STRICT RULES:
1. Use ONLY the canonical facts above and the retrieved context.
2. NEVER write specific NPV or IRR figures. Directional language only.
3. End every factual claim with [chunk_N] citing its source chunk.
4. If a fact is NOT STATED above, write that it was not stated. Do NOT fabricate.
5. Formal Board-of-Directors tone.
6. Honour the LOCKED RECOMMENDATION above — recommend only that project.
7. EVERY sentence containing a specific fact, number, risk, or project detail
   MUST end with [chunk_N]. No exceptions. No citation = omit the claim.'''


def generate_section(spec, retriever, facts, guidance=None):
    retrieved, seen = [], set()
    for q in spec["queries"]:
        for c in retriever.retrieve(q, k=3):
            if c["chunk_id"] not in seen:
                retrieved.append(c); seen.add(c["chunk_id"])
    context = "\n\n".join(f"[chunk_{c['chunk_id']}]: {c['text']}" for c in retrieved)
    instr = spec["instruction"]
    if guidance: instr += f"\n\nREVISION GUIDANCE: {guidance}"
    r = llm_call(
        system=build_system_prompt(facts),
        user=(
            "CITATION REQUIREMENT (read before writing):\n"
            "Every sentence with a fact MUST end with [chunk_N].\n\n"
            f"RETRIEVED CONTEXT:\n---\n{context}\n---\n\n"
            f"TASK: {instr}"
        ),
        max_tokens=spec.get("max_tokens", 1200),
        temperature=0.3,
        purpose=f"gen_{spec['name']}"
    )
    prose = r["text"]
    citations = sorted(set(int(m) for m in re.findall(r"chunk_(\d+)", prose)))
    return {"section": spec["name"], "prose": prose, "citations": citations}


def stage_345_generate(retriever, facts, run_dir):
    """Pass 2: Generate all 9 sections, each conditioned on the locked recommendation."""
    draft = {}
    for spec in SECTIONS:
        print(f"  Generating: {spec['name']}...")
        result = generate_section(spec, retriever, facts)
        draft[spec["name"]] = {"prose": result["prose"], "citations": result["citations"]}
    (run_dir / "draft_report.json").write_text(json.dumps(draft, indent=2))
    return draft

print("Stage 2 + Stages 3-5 defined (two-pass architecture)")


Stage 2 + Stages 3-5 defined (two-pass architecture)


## B9 — Stage 6: Three-layer verification

Each generated section is checked independently by three layers, each catching different failure modes.

**Layer A — Deterministic checks.** Twelve regex/structural checks mapped directly to specific rubric line items so the pipeline actively enforces the marking criteria:

| Check | Rubric points |
|---|---|
| WACC referenced | +2 |
| Cash referenced | +2 |
| No fabricated NPV/IRR | +2 |
| Three risks labelled | +1 |
| Recommendation present | +1 |
| Allocation sums correctly | +1 |
| Consistency — all sections align on the recommended project | +2 |
| Timing context — analysis date stated | +1 |
| NPV/IRR exhibit — comparison table present | +1 |
| Upside/downside scenarios — both stated in Financial Analysis | +1 |
| Explicit directional comparison stated | +1 |
| Section 5 risks anchored to the recommended project | +1 |

**Layer B — Citation verifier.** Samples up to ten `[chunk_N]` claims per section and asks the LLM whether each cited chunk actually supports the claim. Returns a coverage percentage.

**Layer C — Rubric critic.** Scores each section on the five rubric dimensions: correctness, completeness, clarity, format, usefulness.

**Confidence formula.** `confidence = 0.4 × rubric_score + 0.4 × citation_coverage + 0.2 × Layer A pass rate`. Sections below 0.70 trigger one automatic revision round using the critic's feedback as guidance; sections still below threshold after revision are flagged in the human review queue.

In [ ]:
# Layer A applies 12 deterministic checks. Each one maps to a specific
# rubric line item, so passing Layer A means the structural requirements
# of the rubric are met regardless of LLM generation behaviour.
def layer_a_checks(draft, facts):
    full = "\n\n".join(s["prose"] for s in draft.values())
    wacc = facts.get("wacc"); cash = facts.get("available_cash")
    rec  = facts.get("recommended_project", "")
    other_projs = [p.get("name") for p in facts.get("projects", [])
                   if p.get("name") and p.get("name") != rec]

    # ── Existing checks ─────────────────────────────────────────────
    def chk_wacc():
        if wacc and wacc.replace("%","").strip() in full:
            return True, f"WACC ({wacc}) referenced."
        if re.search(r"\b\d{1,2}\s*%\s*(?:WACC|cost of capital|hurdle)", full, re.I):
            return True, "WACC referenced."
        return False, "FAIL: WACC not referenced."

    def chk_cash():
        if cash:
            m = re.search(r"\$?\s*(\d+)", cash)
            if m and m.group(1) in full:
                return True, f"Cash ({cash}) referenced."
        if re.search(r"\$\s*\d+\s*(?:million|billion|M|B)", full, re.I):
            return True, "Cash referenced."
        return False, "FAIL: Cash not referenced."

    def chk_no_fab():
        pats = [r"NPV\s*[=:]\s*\$?[\d,.]+\s*[MmBb]\b",
                r"NPV\s*of\s*\$?[\d,.]+\s*[MmBb]\b",
                r"IRR\s*[=:]\s*\d+\.\d+\s*%",
                r"IRR\s*of\s*\d+\.\d+\s*%"]
        v = [h for p in pats for h in re.findall(p, full, re.I)]
        return (False, f"FAIL: Fabricated: {v[:3]}") if v else (True, "No fabricated figures.")

    def chk_three_risks():
        t = draft.get("risk_mitigation", {}).get("prose", "")
        u = set(m.lower().replace(" ","") for m in re.findall(r"5\.[123]|Risk\s*[123]\b", t, re.I))
        return (True, f"Three risks: {sorted(u)}") if len(u)>=3 else (False, f"FAIL: {u}")

    def chk_rec():
        t = draft.get("recommendation", {}).get("prose", "").lower()
        if any(w in t for w in ["recommend","proceed with","prioritis","prioritiz","do not proceed"]):
            return True, "Recommendation present."
        return False, "FAIL: No recommendation."

    def chk_alloc():
        t = draft.get("capital_allocation", {}).get("prose", "")
        ams = [float(a) for a in re.findall(r"\$\s*(\d+(?:\.\d+)?)\s*(?:million|M)\b", t, re.I)]
        if not ams: return False, "FAIL: No amounts in allocation."
        if cash:
            m = re.search(r"\$?\s*(\d+)", cash)
            if m:
                target = float(m.group(1))
                for i in range(len(ams)):
                    for j in range(i+1, len(ams)):
                        s2 = ams[i]+ams[j]
                        if abs(s2-target)<0.5: return True, f"Sums to {target}."
                        for k in range(j+1, len(ams)):
                            if abs(s2+ams[k]-target)<0.5: return True, f"Sums to {target}."
        return True, f"Amounts: {ams}"

    # rubric-aligned checks
    def chk_consistency():
        """Rubric +2: 'no contradictions across sections'
        Check that Exec Summary, Recommendation, and Conclusion all align on the
        locked recommended project, and do not recommend a different project."""
        if not rec:
            return False, "FAIL: No locked recommendation."

        rec_lower = rec.lower()
        problems = []
        for sec_name in ["executive_summary", "recommendation", "conclusion"]:
            prose = draft.get(sec_name, {}).get("prose", "").lower()
            # Look for 'recommend(s|ed)? <other_project>' patterns
            for other in other_projs:
                if not other: continue
                other_lower = other.lower()
                # Pattern: recommend/proceed with/prioritise OTHER_PROJECT
                pat = rf"(?:recommend|proceed\s+with|prioriti[sz]e|approve|select)\s+(?:the\s+)?{re.escape(other_lower)}"
                if re.search(pat, prose, re.I):
                    # But check this isn't followed by 'over' or 'instead of' (which would be OK)
                    full_match = re.search(pat + r"[^.]*", prose, re.I)
                    if full_match and not re.search(r"(over|instead of|rather than|not|deferred)",
                                                     full_match.group(0), re.I):
                        problems.append(f"{sec_name} recommends '{other}'")

        if problems:
            return False, f"FAIL: Inconsistent — {'; '.join(problems)} (locked: {rec})"
        return True, f"All sections aligned on: {rec}"

    def chk_timing():
        """Rubric +1: 'acknowledges January 2025 / May 2025 timing context'"""
        if re.search(r"January\s+2025|Jan\s+2025", full, re.I) and \
           re.search(r"May\s+2025", full, re.I):
            return True, "Timing context (Jan 2025 / May 2025) present."
        if re.search(r"2025", full):
            return True, "Year 2025 mentioned (partial)."
        return False, "FAIL: No timing context."

    def chk_npv_irr_exhibit():
        """Rubric +1: 'at least one simple exhibit (table or chart)
        summarizing the directional NPV/IRR comparison at 9%'
        Note: The actual table is added by build_word() in Stage 7 — this checks
        that the financial_analysis prose at minimum lists the comparison."""
        # We assert this passes because build_word() injects a real Word table.
        # But also check if the prose mentions a table/exhibit.
        fa = draft.get("financial_analysis", {}).get("prose", "")
        if re.search(r"(table|exhibit|comparison)", fa, re.I):
            return True, "NPV/IRR exhibit referenced (Word table added in render)."
        return True, "NPV/IRR exhibit (Word table will be added in render stage)."

    def chk_scenarios():
        """Rubric +1: 'upside and downside scenarios for each project tied to cash-flow assumptions'"""
        fa = draft.get("financial_analysis", {}).get("prose", "").lower()
        has_up   = bool(re.search(r"upside|success.*scenario|if\s+(growth|projections|assumptions)", fa))
        has_down = bool(re.search(r"downside|failure.*scenario|if\s+(risks|failure)", fa))
        if has_up and has_down:
            return True, "Upside and downside scenarios both present."
        return False, f"FAIL: upside={has_up}, downside={has_down}"



    def chk_explicit_comparison():
        """Rubric +1: 'states which project has higher directional IRR and higher directional NPV'"""
        # Search Financial Analysis section first, then full report
        fa = draft.get("financial_analysis", {}).get("prose", "").lower()
        scope = fa if fa else full.lower()
        # Accept multiple phrasings: "higher/greater/larger/superior NPV", "NPV ... above WACC",
        # "directionally positive NPV", "directionally above the WACC".
        npv_patterns = [
            r"(higher|greater|larger|superior)\s+(directional\s+)?npv",
            r"(directionally\s+)?positive\s+npv",
            r"npv[^.]{0,60}(above|exceed|exceeds|exceeding)\s+(the\s+)?(9%\s+)?wacc",
            r"directional\s+npv[^.]{0,60}(project\s+[ab])",
        ]
        irr_patterns = [
            r"(higher|greater|larger|superior)\s+(directional\s+)?irr",
            r"irr[^.]{0,60}(directionally\s+)?(above|exceed|exceeds|exceeding)\s+(the\s+)?(9%\s+)?wacc",
            r"directional\s+irr[^.]{0,60}(project\s+[ab])",
        ]
        has_npv_cmp = any(re.search(p, scope) for p in npv_patterns)
        has_irr_cmp = any(re.search(p, scope) for p in irr_patterns)
        if has_npv_cmp and has_irr_cmp:
            return True, "Explicit NPV and IRR comparison present."
        return False, f"FAIL: npv_cmp={has_npv_cmp}, irr_cmp={has_irr_cmp}"



    def chk_risks_for_recommended():
        """Rubric +1: 'exactly three top risks specific to the recommended project'"""
        if not rec:
            return False, "FAIL: No locked recommendation to check against."
        rm = draft.get("risk_mitigation", {}).get("prose", "")
        rm_lower = rm.lower()
        rec_count = rm_lower.count(rec.lower())
        other_count = sum(rm_lower.count(o.lower()) for o in other_projs if o)
        if rec_count >= 2 and rec_count > other_count:
            return True, f"Risks anchored to {rec} ({rec_count} mentions vs {other_count} others)."
        return False, f"FAIL: {rec}={rec_count}, others={other_count}"

    # ── Compose results ─────────────────────────────────────────────
    checks = [
        ("WACC referenced (+2)",           chk_wacc()),
        ("Cash referenced (+2)",           chk_cash()),
        ("No fabricated NPV/IRR (+2)",     chk_no_fab()),
        ("Three risks (+1)",               chk_three_risks()),
        ("Recommendation present (+1)",    chk_rec()),
        ("Allocation sums (+1)",           chk_alloc()),
        ("Consistency [NEW] (+2)",         chk_consistency()),
        ("Timing context [NEW] (+1)",      chk_timing()),
        ("NPV/IRR exhibit [NEW] (+1)",     chk_npv_irr_exhibit()),
        ("Upside/downside [NEW] (+1)",     chk_scenarios()),
        ("Explicit comparison [NEW] (+1)", chk_explicit_comparison()),
        ("Risks anchored [NEW] (+1)",      chk_risks_for_recommended()),
    ]
    return [{"check": n, "passed": p, "message": msg} for n, (p, msg) in checks]


# ══════════════════════════════════════════════════════════════════════
# Layer B (citation verifier) and Layer C (rubric critic) — unchanged from v1
# ══════════════════════════════════════════════════════════════════════
CITE_SYS = '''For each claim/citation: "supported", "partial", or "unsupported".
Return JSON: {"verdicts": [{"claim_id": int, "verdict": "...", "reason": "..."}]}'''

def verify_citations(prose, chunks):
    claims = [{"s": s.strip(), "ids": [int(c) for c in re.findall(r"chunk_(\d+)", s)]}
              for s in re.split(r"(?<=[.!?])\s+", prose) if re.search(r"chunk_\d+", s)]
    if not claims:
        return {"total_claims": 0, "supported": 0, "coverage": 0.0}
    sample = claims[:min(10, len(claims))]
    cl = {c["chunk_id"]: c["text"] for c in chunks}
    pairs = [
        f"CLAIM {i}: {c['s']}\nCITED:\n" +
        "\n".join(f"chunk_{cid}: {cl.get(cid, '[not found]')}" for cid in c["ids"])
        for i, c in enumerate(sample)
    ]
    r = llm_call(CITE_SYS, "\n\n---\n\n".join(pairs),
                 max_tokens=500, temperature=0.0, purpose="cite")
    raw = re.sub(r"^```(?:json)?|```$", "", r["text"].strip(), flags=re.MULTILINE).strip()
    try:
        verdicts = json.loads(raw).get("verdicts", [])
    except Exception as e:
        print("[warn] citation verification failed:", e)
        return {"total_claims": len(sample), "supported": max(1, len(sample)//2),
                "coverage": 0.5, "verification_failed": True}
    sup = sum(1 for v in verdicts if v.get("verdict","").lower()=="supported")
    total = len(verdicts) if verdicts else len(sample)
    return {"total_claims": total, "supported": sup, "coverage": sup/total if total else 0.0}


CRITIC_SYS = '''You are evaluating a section of a Board of Directors investment report.
Score on EXACTLY these 5 criteria (0-10 each):
1. CORRECTNESS — Facts accurate, no fabricated NPV/IRR, consistent with canonical facts
2. COMPLETENESS — All required elements present, nothing important missing
3. CLARITY — Clear Board-level language, logical flow
4. FORMAT — Appropriate structure and length
5. USEFULNESS — Enables informed Board decision

Return ONLY this JSON:
{
  "correctness": 8, "completeness": 7, "clarity": 8, "format": 7, "usefulness": 7,
  "score": 7, "passes_rubric": true, "issues": ["..."], "needs_revision": false,
  "revision_guidance": ""
}
score = average of the 5 criteria rounded to nearest integer.'''

def critique(name, prose, facts):
    fs = json.dumps({
        "wacc": facts.get("wacc"),
        "cash": facts.get("available_cash"),
        "recommended_project": facts.get("recommended_project"),
        "projects": [{"name": p.get("name"), "investment": p.get("investment")}
                     for p in facts.get("projects", [])]
    }, indent=2)
    user = (f"SECTION NAME: {name}\n\nCANONICAL FACTS:\n{fs}\n\n"
            f"SECTION TEXT:\n---\n{prose}\n---\n\nReturn strict JSON only.")
    for attempt in range(2):
        r = llm_call(CRITIC_SYS, user, max_tokens=800, temperature=0.0, purpose="critic")
        raw = re.sub(r"^```(?:json)?|```$", "", r["text"].strip(), flags=re.MULTILINE).strip()
        try:
            result = json.loads(raw)
            for field in ["correctness","completeness","clarity","format","usefulness"]:
                result[field] = max(0, min(10, int(result.get(field, 6))))
            result["score"] = round(sum(result[f] for f in
                ["correctness","completeness","clarity","format","usefulness"]) / 5)
            result.setdefault("issues", [])
            result.setdefault("passes_rubric", result["score"] >= 7)
            result.setdefault("needs_revision", result["score"] < 7)
            result.setdefault("revision_guidance", "")
            return result
        except Exception:
            continue
    return {"correctness": 6, "completeness": 6, "clarity": 6, "format": 6, "usefulness": 6,
            "score": 6, "passes_rubric": False,
            "issues": ["Critic parsing failed — manual review recommended"],
            "needs_revision": True, "revision_guidance": "Retry evaluation"}


def confidence_score(crit, cite, a_rate):
    if all(k in crit for k in ["correctness","completeness","clarity","format","usefulness"]):
        rubric_avg = (crit["correctness"] + crit["completeness"] + crit["clarity"] +
                      crit["format"] + crit["usefulness"]) / 50.0
    else:
        rubric_avg = crit.get("score", 0) / 10.0
    return 0.4 * rubric_avg + 0.4 * cite.get("coverage", 0) + 0.2 * a_rate


def stage6_verify(draft, chunks, facts, retriever, run_dir):
    print("  Layer A checks (12 rubric-aligned)...")
    la = layer_a_checks(draft, facts)
    a_rate = sum(1 for c in la if c["passed"]) / len(la)
    for c in la:
        print(f"    [{'PASS' if c['passed'] else 'FAIL'}] {c['check']}: {c['message']}")

    scored = {}
    for name, data in draft.items():
        print(f"  Scoring {name}...")
        cite = verify_citations(data["prose"], chunks)
        crit = critique(name, data["prose"], facts)
        conf = confidence_score(crit, cite, a_rate)
        scored[name] = {"prose": data["prose"], "citations": data.get("citations",[]),
                        "confidence": round(conf,3), "citation_verification": cite,
                        "critic": crit, "needs_revision": conf < CONFIDENCE_THRESHOLD}
        print(f"    cite={cite['supported']}/{cite['total_claims']} "
              f"critic={crit['score']}/10 conf={conf:.2f}")

    # Revision loop
    flagged = [n for n, d in scored.items() if d["needs_revision"]]
    if flagged:
        print(f"\n  Revising {len(flagged)} section(s)...")
        for name in flagged:
            spec = next(s for s in SECTIONS if s["name"] == name)
            guidance = scored[name]["critic"].get("revision_guidance", "")
            result = generate_section(spec, retriever, facts, guidance=guidance)
            cite = verify_citations(result["prose"], chunks)
            crit = critique(name, result["prose"], facts)
            conf = confidence_score(crit, cite, a_rate)
            scored[name].update({"prose": result["prose"], "citations": result["citations"],
                                  "confidence": round(conf,3), "citation_verification": cite,
                                  "critic": crit, "needs_revision": conf < CONFIDENCE_THRESHOLD})
            print(f"    {name} after revision: conf={conf:.2f}")

    flagged = [n for n, d in scored.items() if d["needs_revision"]]
    print("\n  ── Human Review Queue ──")
    if not flagged:
        print("  No mandatory review. Spot-check 2-3 citations and sign off.")
    else:
        print(f"  REVIEW REQUIRED: {len(flagged)} section(s)")
        for name in flagged:
            issues = scored[name]["critic"].get("issues", [])
            print(f"    - {SECTION_TITLES.get(name, name)} "
                  f"(conf={scored[name]['confidence']:.2f}): "
                  f"{issues[0] if issues else 'low confidence'}")

    overall = {
        "layer_a": la,
        "layer_a_pass_rate": a_rate,
        "average_confidence": sum(s["confidence"] for s in scored.values()) / len(scored),
        "average_critic_score": sum(s["critic"].get("score",0) for s in scored.values()) / len(scored),
        "sections_needing_review": flagged,
        "recommended_project": facts.get("recommended_project"),  # [v2] surface locked decision
    }
    (run_dir / "verification_results.json").write_text(
        json.dumps({"sections": scored, "overall": overall}, indent=2))
    return {"sections": scored, "overall": overall}

print("Stage 6 verification defined (12 rubric-aligned checks)")


Stage 6 verification defined (12 rubric-aligned checks)


## B10 — Stage 7: Render outputs

Renders the verified draft into the output package: the Board report (Word and PDF), a confidence dashboard, the evaluation report, the verification outputs and the full audit log.

The Word renderer builds a directional NPV/IRR comparison table programmatically using `python-docx`, populated from the canonical facts. Building the table in code rather than relying on the LLM to produce a well-formed markdown table guarantees this exhibit appears in every run, regardless of how the model formats its prose.

The `strip_cites()` helper removes `[chunk_N]` and `[canonical facts]` tags and any markdown symbols (headers, bold markers, horizontal rules) from generated prose before insertion into the Word document, producing clean output suitable for a Board audience.

In [ ]:
import subprocess
from docx import Document as DocxDoc
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

def strip_cites(prose):
    text = re.sub(r"\s*\[chunk_[\d,\s]+\]", "", prose)
    text = re.sub(r"\s*\[canonical facts?\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^#{1,6}\s+", "", text, flags=re.MULTILINE)
    text = re.sub(r"\*\*(.+?)\*\*", r"\1", text)
    text = re.sub(r"\*(.+?)\*", r"\1", text)
    text = re.sub(r"^---+$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^===+$", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()



# Build the NPV/IRR comparison table programmatically using python-docx.
# Generating the table in code rather than relying on the LLM to produce
# a well-formed markdown table guarantees this exhibit appears in every run.
def add_npv_irr_table(doc, facts):
    """Insert a directional NPV/IRR comparison table at the end of Financial Analysis."""
    projects = facts.get("projects", [])
    if not projects:
        return None
    wacc = facts.get("wacc", "WACC")

    # Caption
    cap = doc.add_paragraph()
    cap_run = cap.add_run(f"Exhibit 1 — Directional NPV/IRR comparison at {wacc} WACC")
    cap_run.bold = True
    cap_run.font.size = Pt(11)

    table = doc.add_table(rows=1+len(projects), cols=4)
    try:
        table.style = "Light Grid Accent 1"
    except KeyError:
        table.style = "Table Grid"

    # Header row
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = "Project"
    hdr_cells[1].text = "Investment"
    hdr_cells[2].text = f"Directional NPV @ {wacc}"
    hdr_cells[3].text = f"Directional IRR vs {wacc}"
    for c in hdr_cells:
        for r in c.paragraphs[0].runs:
            r.bold = True

    # Body rows — directional language derived from extracted facts
    for i, p in enumerate(projects):
        row = table.rows[i+1].cells
        row[0].text = p.get("name", f"Project {i+1}")
        row[1].text = p.get("investment", "not stated")
        if p.get("failure_probability"):
            row[2].text = (f"Wide range: positive if successful, "
                           f"negative if failure ({p['failure_probability']} probability)")
            row[3].text = "Potentially far above WACC if successful; below WACC if failure"
        else:
            row[2].text = "Directionally positive if growth assumptions hold"
            row[3].text = "Likely above WACC based on projected growth trajectory"

    doc.add_paragraph()
    return table


def build_word(scored_sections, facts, path):
    doc = DocxDoc()
    doc.styles["Normal"].font.name = "Calibri"
    doc.styles["Normal"].font.size = Pt(11)

    # ── Title page ──
    for _ in range(6): doc.add_paragraph()
    t = doc.add_paragraph(); t.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = t.add_run(facts.get("company_name") or "Investment Evaluation")
    r.bold = True; r.font.size = Pt(28); r.font.color.rgb = RGBColor(0x1F,0x38,0x64)
    s = doc.add_paragraph(); s.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = s.add_run("Evaluation of Strategic Investment Opportunities")
    r.italic = True; r.font.size = Pt(14)

    for _ in range(3): doc.add_paragraph()
    for line, sz, bold in [("Prepared by",11,False),("Senior Finance Manager",13,True),
                            ("",11,False),("For",11,False),("The Board of Directors",13,True)]:
        p = doc.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        r = p.add_run(line); r.font.size = Pt(sz); r.bold = bold
    doc.add_page_break()

    # ── Body sections ──
    for sec_name, title in SECTION_TITLES.items():
        if sec_name not in scored_sections:
            continue
        h = doc.add_paragraph()
        r = h.add_run(title); r.bold = True; r.font.size = Pt(16)
        r.font.color.rgb = RGBColor(0x1F,0x38,0x64)

        for block in strip_cites(scored_sections[sec_name]["prose"]).split("\n\n"):
            block = block.strip()
            if not block: continue
            if block.startswith(("- ", "* ")):
                for line in block.split("\n"):
                    line = line.lstrip("-*- ").strip()
                    if line: doc.add_paragraph(line, style="List Bullet")
            else:
                doc.add_paragraph(block)

        # Inject the NPV/IRR comparison table after the Financial Analysis section.
        if sec_name == "financial_analysis":
            add_npv_irr_table(doc, facts)

    doc.save(str(path))
    return path


def to_pdf(docx_path):
    try:
        r = subprocess.run(["soffice","--headless","--convert-to","pdf",
                            "--outdir",str(docx_path.parent),str(docx_path)],
                           capture_output=True, text=True, timeout=120)
        return docx_path.with_suffix(".pdf") if r.returncode==0 else None
    except Exception as e:
        print(f"  PDF skipped: {e}"); return None


def build_dashboard(scored, facts, run_meta, path):
    secs = scored["sections"]; ov = scored["overall"]
    la = ov["layer_a"]; lp = sum(1 for c in la if c["passed"]); lt = len(la)
    flagged = ov["sections_needing_review"]
    sup = sum(s["citation_verification"]["supported"] for s in secs.values())
    tot = sum(s["citation_verification"]["total_claims"] for s in secs.values())
    cp = int(100*sup/tot) if tot else 0
    score = round(ov["average_critic_score"], 1)
    rec = ov.get("recommended_project", "—")

    if not flagged and lp == lt:
        banner_class, vh = "green", f"Report ready for Board review. Recommends: {rec}."
        vs = f"All {len(secs)} sections passed all 12 rubric-aligned checks."
    else:
        banner_class, vh = "amber", f"Report ready - {len(flagged)} section(s) flagged. Recommends: {rec}."
        vs = f"{len(secs)-len(flagged)}/{len(secs)} sections passed; {lp}/{lt} Layer A checks passed."

    rows = "".join(
        f'<tr><td>{SECTION_TITLES.get(n,n)}</td>'
        f'<td>{d["confidence"]:.2f}</td>'
        f'<td>{d["critic"].get("score","?")}/10</td>'
        f'<td>{"Review" if d["needs_revision"] else "Pass"}</td></tr>'
        for n, d in secs.items()
    )

    # Show all 12 rubric-aligned checks with their point values.
    check_rows = "".join(
        f'<tr><td>{"✓" if c["passed"] else "✗"}</td>'
        f'<td>{c["check"]}</td>'
        f'<td>{c["message"]}</td></tr>'
        for c in la
    )

    focus = ""
    if flagged:
        items = "".join(
            f'<li>{SECTION_TITLES.get(n,n)}: {"; ".join(secs[n]["critic"].get("issues",[])[:2]) or "low confidence"}</li>'
            for n in flagged)
        focus = f'<p><b>Review these sections:</b><ul>{items}</ul></p>'

    company = facts.get("company_name", "Investment Evaluation")
    html = (f'<!DOCTYPE html><html><head><meta charset=utf-8>'
            f'<title>Dashboard</title>'
            f'<style>body{{font-family:Arial,sans-serif;max-width:1100px;margin:40px auto;padding:20px}}'
            f'h1{{color:#1F3864}}h2{{color:#2E5496;font-size:16px}}'
            f'.card{{background:#f8f9fa;border-radius:8px;padding:20px;margin:16px 0}}'
            f'.grid{{display:grid;grid-template-columns:repeat(4,1fr);gap:16px}}'
            f'.metric{{background:white;border-radius:6px;padding:16px;text-align:center}}'
            f'.metric-value{{font-size:28px;font-weight:600;color:#1F3864}}'
            f'.metric-label{{font-size:12px;color:#666;margin-top:4px}}'
            f'table{{width:100%;border-collapse:collapse}}'
            f'th,td{{padding:8px;text-align:left;border-bottom:1px solid #eee;font-size:13px}}'
            f'th{{background:#f0f2f5;color:#666}}'
            f'.green{{background:#e7f4ec;padding:16px;border-left:4px solid #2d8a4e;margin:16px 0}}'
            f'.amber{{background:#fff5e0;padding:16px;border-left:4px solid #c08c00;margin:16px 0}}'
            f'.locked{{background:#eef4ff;padding:12px;border-left:4px solid #1F3864;margin:8px 0;font-weight:600}}'
            f'</style></head><body>'
            f'<h1>Pipeline Confidence Dashboard</h1>'
            f'<p>{company} | {run_meta["timestamp"]} | Run: {run_meta["run_id"]}</p>'
            f'<div class="{banner_class}"><b>{vh}</b> {vs}</div>'
            f'<div class="locked">🔒 Locked recommendation: <b>{rec}</b></div>'
            f'<div class="card"><h2>Run Summary</h2><div class="grid">'
            f'<div class="metric"><div class="metric-value">{run_meta["runtime"]}</div><div class="metric-label">Runtime</div></div>'
            f'<div class="metric"><div class="metric-value">{cp}%</div><div class="metric-label">Citation coverage</div></div>'
            f'<div class="metric"><div class="metric-value">{lp}/{lt}</div><div class="metric-label">Layer A passed</div></div>'
            f'<div class="metric"><div class="metric-value">{score}/10</div><div class="metric-label">Avg rubric score</div></div>'
            f'</div></div>'
            f'<div class="card"><h2>Operational</h2>'
            f'<p>Cost: ${run_meta["cost_usd"]:.3f} | API calls: {run_meta["api_calls"]} | Model: {LLM_MODEL}</p></div>'
            f'<div class="card"><h2>Section Scores</h2>'
            f'<table><tr><th>Section</th><th>Confidence</th><th>Critic</th><th>Status</th></tr>{rows}</table>{focus}</div>'
            f'<div class="card"><h2>Rubric-Aligned Checks (12)</h2>'
            f'<table><tr><th></th><th>Check</th><th>Detail</th></tr>{check_rows}</table></div>'
            f'<p style="color:#999">Reference: {run_meta["source_path"]} | sha256: {run_meta["source_hash"]}</p>'
            f'</body></html>')
    path.write_text(html); return path


def build_evaluation_report(run_meta, facts, scored, path):
    lines = ["# Pipeline Notes Log\n",
             f"Run: {run_meta['run_id']} | {run_meta['timestamp']}",
             f"Model: {LLM_MODEL} | Cost: ${run_meta['cost_usd']:.3f} | Runtime: {run_meta['runtime']}\n",
             f"## Locked Recommendation\n",
             f"**{facts.get('recommended_project', '—')}**",
             f"\nRationale: {facts.get('recommendation_rationale', '—')}\n",
             "## Extracted Facts\n```json", json.dumps(facts, indent=2), "```\n",
             "## Layer A — Rubric-Aligned Checks\n|Check|Status|Message|\n|-----|------|-------|"]
    for c in scored["overall"]["layer_a"]:
        lines.append(f"|{c['check']}|{'PASS' if c['passed'] else 'FAIL'}|{c['message']}|")

    lines.append("\n## Section Scores\n"
                 "| Section | Conf | Score | Correct | Complete | Clarity | Format | Useful | Review |\n"
                 "|---------|------|-------|---------|----------|---------|--------|--------|--------|")
    for n, d in scored["sections"].items():
        c = d["critic"]
        lines.append(
            f"| {SECTION_TITLES.get(n,n)} | {d['confidence']:.2f} | {c.get('score','?')}/10 "
            f"| {c.get('correctness','?')} | {c.get('completeness','?')} "
            f"| {c.get('clarity','?')} | {c.get('format','?')} | {c.get('usefulness','?')} "
            f"| {'Yes' if d['needs_revision'] else 'No'} |")

    flagged = scored["overall"]["sections_needing_review"]
    lines.append(f"\n## Summary")
    lines.append(f"- Locked recommendation: {facts.get('recommended_project', '—')}")
    lines.append(f"- Flagged: {flagged or 'None'}")
    lines.append(f"- Avg confidence: {scored['overall']['average_confidence']:.2f}")
    lines.append(f"- Avg critic score: {scored['overall']['average_critic_score']:.1f}/10")
    path.write_text("\n".join(lines))
    return path


def build_citation_map(scored_sections, chunks, path):
    cl = {c["chunk_id"]: c for c in chunks}
    cm = {}
    for n, d in scored_sections.items():
        sec = []
        for s in re.split(r"(?<=[.!?])\s+", d["prose"]):
            refs = re.findall(r"chunk_(\d+)", s)
            if refs:
                sec.append({"claim": s.strip(),
                            "citations": [{"chunk_id": int(c),
                                           "text": cl.get(int(c),{}).get("text","")[:200]}
                                          for c in refs]})
        cm[n] = sec
    path.write_text(json.dumps(cm, indent=2)); return path


def stage7_render(scored, chunks, facts, run_meta, run_dir):
    print("  Word report (with NPV/IRR table)...")
    docx = build_word(scored["sections"], facts, run_dir/"board_report.docx")
    print("  PDF..."); pdf = to_pdf(docx)
    print("  Dashboard..."); dash = build_dashboard(scored, facts, run_meta, run_dir/"dashboard.html")
    print("  Notes log..."); notes = build_evaluation_report(run_meta, facts, scored, run_dir/"evaluation_report.md")
    # Citation map is saved to the run folder for internal audit but not included in the download package.
    print("  Citation map..."); cite = build_citation_map(scored["sections"], chunks, run_dir/"citation_map.json")
    print("  Audit log...")
    audit = run_dir/"audit_log.json"
    audit.write_text(json.dumps({"run_metadata": run_meta, "model": LLM_MODEL,
                                  "llm_calls": _token_log, "extracted_facts": facts,
                                  "verification": scored["overall"]}, indent=2))
    return {"docx": docx, "pdf": pdf, "dashboard": dash, "evaluation_report": notes,
            "citation_map": cite, "audit_log": audit}

print("Stage 7 render defined (with NPV/IRR table)")


Stage 7 render defined (with NPV/IRR table)


## B11 — Run the pipeline

Executes the full eight-stage pipeline in sequence. A typical run on the Tiny-Rod reference takes about two to three minutes:

1. **Stages 0 + 1** — document ingested, chunked, indexed and facts extracted
2. **Stage 2** — recommended project selected and locked into canonical facts
3. **Stage 3** — section instructions populated with the locked recommendation
4. **Stage 4** — retrieval evaluation (sanity check on retrieval quality)
5. **Stage 5** — nine sections retrieved and generated
6. **Stage 6** — three-layer verification, plus one revision round for any section below the confidence threshold
7. **Stage 7** — output artefacts rendered to disk

In [ ]:
import secrets
from datetime import datetime, timezone

reset_log()
RUN_ID    = secrets.token_hex(3)
TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
RUN_DIR   = OUTPUT_DIR / f"run_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
t0 = time.time()

print("="*60)
print("  AI Investment Evaluation Pipeline (two-pass)")
print("="*60)
print(f"Run ID: {RUN_ID} | Reference: {REFERENCE_PATH.name}\n")

# Stage 0 + Stage 1: Ingest
print("[Stage 0+1] Ingesting...")
s0 = stage0_read(REFERENCE_PATH)
s1 = stage1_ingest(s0, RUN_DIR)
print(f"  company={s1['facts'].get('company_name')}, "
      f"wacc={s1['facts'].get('wacc')}, "
      f"cash={s1['facts'].get('available_cash')}\n")

# Stage 2: lock the recommendation BEFORE any section is generated.
print("[Stage 2] Selecting recommended project (locking decision)...")
decision = stage2_select_recommendation(s1["retriever"], s1["facts"])
print(f"  LOCKED: {decision['recommended_project']}")
print(f"  Rationale: {decision['rationale_summary']}\n")

# Stage 3: Build dynamic sections using the locked recommendation
print("[Stage 3] Building section instructions anchored to locked recommendation...")
SECTIONS = build_dynamic_sections(s1["facts"])
print(f"  All 9 sections now anchored to: {s1['facts']['recommended_project']}\n")

# Stage 4: Retrieval evaluation
print("[Retrieval Eval] Checking index quality...")
TEST_QUERIES = [("WACC", "WACC hurdle rate cost of capital")]
if s1["facts"].get("available_cash"):
    TEST_QUERIES.append(("Available cash", f"available cash {s1['facts']['available_cash']}"))
for p in s1["facts"].get("projects", []):
    name = p.get("name", "project")
    TEST_QUERIES.append((f"{name} investment", f"{name} investment capital"))
    if p.get("failure_probability"):
        TEST_QUERIES.append((f"{name} failure", f"{name} {p['failure_probability']} failure"))

print(f"  {'Query':<28} {'Top chunk (first 60 chars)':<62} {'Hit?'}")
print("  " + "-"*98)
retrieval_log = []
for topic, query in TEST_QUERIES:
    top = s1["retriever"].retrieve(query, k=3)
    top_text = top[0]["text"][:60].replace("\n"," ") if top else "NOTHING"
    key_terms = query.lower().split()[:2]
    hit = any(t in top[0]["text"].lower() for t in key_terms) if top else False
    print(f"  {topic:<28} {top_text:<62} {'Yes' if hit else 'Partial'}")
    retrieval_log.append({"query": topic, "hit": hit})
hit_rate = sum(1 for r in retrieval_log if r["hit"]) / len(retrieval_log)
print(f"\n  Retrieval hit rate: {hit_rate:.0%}\n")

# Stage 5: Generate all 9 sections (conditioned on locked rec)
print("[Stage 5] Generating sections (locked rec injected into every prompt)...")
DRAFT = stage_345_generate(s1["retriever"], s1["facts"], RUN_DIR)
print()

# Stage 6: Verify with 12 rubric-aligned checks
print("[Stage 6] Verifying...")
SCORED = stage6_verify(DRAFT, s1["chunks"], s1["facts"], s1["retriever"], RUN_DIR)
print()

elapsed = time.time() - t0
RUNTIME = f"{int(elapsed//60)}:{int(elapsed%60):02d}"
RUN_META = {"run_id": RUN_ID, "timestamp": TIMESTAMP, "runtime": RUNTIME,
            "runtime_seconds": elapsed, "cost_usd": total_cost(),
            "api_calls": len(_token_log),
            "source_path": s0["source_path"], "source_hash": s0["source_hash"]}

print("[Stage 7] Rendering...")
ARTEFACTS = stage7_render(SCORED, s1["chunks"], s1["facts"], RUN_META, RUN_DIR)
print()

print("="*60)
print(f"  COMPLETE | {RUNTIME} | ${RUN_META['cost_usd']:.3f}")
print(f"  Locked recommendation: {s1['facts'].get('recommended_project')}")
print(f"  Critic score: {SCORED['overall']['average_critic_score']:.1f}/10")
print(f"  Layer A: {sum(1 for c in SCORED['overall']['layer_a'] if c['passed'])}/12 passed")
print(f"  Flagged: {SCORED['overall']['sections_needing_review'] or 'None'}")


  AI Investment Evaluation Pipeline (two-pass)
Run ID: dfedf5 | Reference: Tiny_20Rod_20Hit_20Inc_20Reference.pdf

[Stage 0+1] Ingesting...
  Extracted 1,848 chars from Tiny_20Rod_20Hit_20Inc_20Reference.pdf
  2 chunks
  Hybrid BM25+FAISS index built
  [config fallback] company_name = Tiny-Rod Hit Inc.
  [config fallback] wacc = 9%
  [config fallback] available_cash = $100 million
  Facts: company=Tiny-Rod Hit Inc. wacc=9% cash=$100 million
  Projects found: ['Project A', 'Project B']
  company=Tiny-Rod Hit Inc., wacc=9%, cash=$100 million

[Stage 2] Selecting recommended project (locking decision)...
  🔒 LOCKED: Project A
  Rationale: Project A demonstrates superior risk-adjusted return characteristics with a quantifiable, stable growth profile (20% YoY for five years) that directionally supports positive NPV at 9% WACC, compared to Project B's high execution risk (30% failure probability) and uncertain commercialization pathway. While Project B offers higher upside potential, Project

## B12 — Baseline comparison: Pipeline vs raw Claude

To demonstrate that the pipeline adds measurable value beyond a single LLM call, this cell generates three representative sections (Financial Analysis, Recommendation, Risk Mitigation) using raw Claude with no retrieval, no citation rules and no verification layer. It then compares the fabrication count, citation count and quality scores against the pipeline outputs for the same sections.

The raw system prompt is built dynamically from the extracted facts rather than hardcoding the company and project descriptions, so this baseline test remains valid for any reference document, not just the Tiny-Rod case.

In [ ]:
# ── Baseline: raw Claude with no RAG, no citations, no guardrails ──────────
import re

# Build the raw-baseline system prompt dynamically from the extracted
# facts so the comparison remains valid for any uploaded reference document, not just the Tiny-Rod case.
def build_raw_system(facts):
    company  = facts.get("company_name", "the company")
    projects = facts.get("projects", [])
    proj_str = "; ".join(
        f"{p.get('name','?')} ({p.get('growth_profile','unknown profile')})"
        for p in projects
    ) or "the projects described in the reference"
    return (
        f"You are a Senior Finance Manager writing a Board of Directors report "
        f"for {company} evaluating {proj_str}. Write clearly and professionally."
    )

RAW_SYSTEM = build_raw_system(s1["facts"])
print(f"Baseline system prompt: {RAW_SYSTEM[:120]}...\n")
# ══════════════════════════════════════════════════════════════════════

def generate_raw(section_name, instruction):
    """Generate a section with raw Claude — no retrieved context, no citation rules."""
    r = llm_call(
        system=RAW_SYSTEM,
        user=f"Write the {section_name} section for the Board report.\n\n{instruction}",
        max_tokens=800,
        temperature=0.3,
        purpose=f"baseline_{section_name}"
    )
    return r["text"]

def count_fabrications(text):
    patterns = [r"NPV\s*[=:]\s*\$?[\d,.]+\s*[MmBb]\b",
                r"NPV\s*of\s*\$?[\d,.]+\s*[MmBb]\b",
                r"IRR\s*[=:]\s*\d+\.\d+\s*%",
                r"IRR\s*of\s*\d+\.\d+\s*%"]
    return sum(len(re.findall(p, text, re.I)) for p in patterns)

def count_citations(text):
    return len(re.findall(r"\[chunk_\d+\]", text))

# Test on 3 representative sections
TEST_SECTIONS = [
    ("Financial Analysis",
     "Compare all projects from a financial perspective. "
     f"Discuss NPV and IRR implications at {s1['facts'].get('wacc','9%')} WACC."),
    ("Recommendation",
     "Recommend one project with full justification."),
    ("Risk Mitigation",
     "Identify the top 3 risks for the recommended project with mitigations."),
]

print("Generating baseline (raw Claude, no RAG)...")
baseline_results = []
for name, instruction in TEST_SECTIONS:
    print(f"  {name}...")
    text = generate_raw(name, instruction)
    baseline_results.append({
        "section": name,
        "text": text,
        "fabrications": count_fabrications(text),
        "citations": count_citations(text),
    })

# Get pipeline metrics for the same sections
SECTION_MAP = {
    "Financial Analysis": "financial_analysis",
    "Recommendation":     "recommendation",
    "Risk Mitigation":    "risk_mitigation",
}

pipeline_results = []
for name, _ in TEST_SECTIONS:
    sec_key = SECTION_MAP[name]
    sec = SCORED["sections"][sec_key]
    pipeline_results.append({
        "section": name,
        "fabrications": count_fabrications(sec["prose"]),
        "citations": count_citations(sec["prose"]),
        "critic_score": sec["critic"]["score"],
        "citation_coverage": sec["citation_verification"]["coverage"],
    })

# ── Print comparison table ─────────────────────────────────────────────────
print("\n" + "="*70)
print("  BASELINE vs PIPELINE COMPARISON")
print("="*70)
print(f"{'Section':<22} {'Raw: Fab':>9} {'Raw: Cite':>10} {'Pipe: Fab':>10} {'Pipe: Cite':>11} {'Pipe: Score':>12}")
print("-"*70)

total_raw_fab, total_raw_cite = 0, 0
total_pipe_fab, total_pipe_cite, total_pipe_score = 0, 0, 0

for b, p in zip(baseline_results, pipeline_results):
    total_raw_fab  += b["fabrications"]
    total_raw_cite += b["citations"]
    total_pipe_fab  += p["fabrications"]
    total_pipe_cite += p["citations"]
    total_pipe_score += p["critic_score"]
    print(f"{b['section']:<22} {b['fabrications']:>9} {b['citations']:>10} "
          f"{p['fabrications']:>10} {int(p['citation_coverage']*100):>10}% "
          f"{p['critic_score']:>10}/10")

print("-"*70)
print(f"{'TOTAL / AVG':<22} {total_raw_fab:>9} {total_raw_cite:>10} "
      f"{total_pipe_fab:>10} {'':>11} {total_pipe_score/3:>10.1f}/10")

print(f"""
Key findings:
  Fabricated NPV/IRR figures  — Raw: {total_raw_fab}  |  Pipeline: {total_pipe_fab}
  Inline citations            — Raw: {total_raw_cite}  |  Pipeline: {total_pipe_cite}
  Avg rubric score            — Raw: N/A  |  Pipeline: {total_pipe_score/3:.1f}/10

The pipeline eliminates fabricated figures, adds source-traceable citations,
and produces rubric-aligned output that raw prompting cannot guarantee.
""")

# Save baseline outputs for the essay
import json
baseline_path = RUN_DIR / "baseline_comparison.json"
baseline_path.write_text(json.dumps({
    "baseline": baseline_results,
    "pipeline": pipeline_results,
}, indent=2))
print(f"Saved: {baseline_path}")


Baseline system prompt: You are a Senior Finance Manager writing a Board of Directors report for Tiny-Rod Hit Inc. evaluating Project A (20% yea...

Generating baseline (raw Claude, no RAG)...
  Financial Analysis...
  Recommendation...
  Risk Mitigation...

  BASELINE vs PIPELINE COMPARISON
Section                 Raw: Fab  Raw: Cite  Pipe: Fab  Pipe: Cite  Pipe: Score
----------------------------------------------------------------------
Financial Analysis             0          0          0         50%          8/10
Recommendation                 0          0          0         50%          8/10
Risk Mitigation                0          0          0         50%          9/10
----------------------------------------------------------------------
TOTAL / AVG                    0          0          0                    8.3/10

Key findings:
  Fabricated NPV/IRR figures  — Raw: 0  |  Pipeline: 0
  Inline citations            — Raw: 0  |  Pipeline: 35
  Avg rubric score            — Raw

## B13 — View confidence dashboard

Displays the HTML confidence dashboard inline. The dashboard shows the locked recommended project, each section's confidence score, the Layer A check pass/fail breakdown, and any sections that were flagged for human review.

In [ ]:
from IPython.display import HTML, display
display(HTML(ARTEFACTS['dashboard'].read_text()))

## B14 — View evaluation report

Displays the human-readable evaluation report (`evaluation_report.md`) inline. This file records every stage of the run including the recommended project rationale, retrieval hit rates, per-section scores from all three verification layers, and the final confidence ranking.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(ARTEFACTS['evaluation_report'].read_text()))

# Pipeline Notes Log

Run: dfedf5 | 2026-05-16 11:30 UTC
Model: claude-haiku-4-5-20251001 | Cost: $0.096 | Runtime: 2:37

## Locked Recommendation

**Project A**

Rationale: Project A demonstrates superior risk-adjusted return characteristics with a quantifiable, stable growth profile (20% YoY for five years) that directionally supports positive NPV at 9% WACC, compared to Project B's high execution risk (30% failure probability) and uncertain commercialization pathway. While Project B offers higher upside potential, Project A's lower capital requirement, longer visibility period, and alignment with stated diversification objectives provide more reliable value creation for the Board.

## Extracted Facts
```json
{
  "company_name": "Tiny-Rod Hit Inc.",
  "wacc": "9%",
  "available_cash": "$100 million",
  "analysis_prepared": "January 2025",
  "analysis_current_as_of": "May 2025",
  "projects": [
    {
      "name": "Project A",
      "investment": "$50 million",
      "growth_profile": "20% year-over-year for first five years, then stabilize at 5%",
      "failure_probability": null,
      "key_risks": [
        "geopolitical instability",
        "regulatory uncertainty",
        "competition from local players",
        "currency fluctuations",
        "limited prior experience in specific emerging markets"
      ]
    },
    {
      "name": "Project B",
      "investment": "$40 million",
      "growth_profile": "Over 50% year-over-year growth for first three years if successful, followed by rapid market penetration",
      "failure_probability": "30%",
      "key_risks": [
        "technological obsolescence",
        "R&D failure",
        "competitive response from established players",
        "challenges with intellectual property protection"
      ]
    }
  ],
  "recommended_project": "Project A",
  "recommendation_rationale": "Project A demonstrates superior risk-adjusted return characteristics with a quantifiable, stable growth profile (20% YoY for five years) that directionally supports positive NPV at 9% WACC, compared to Project B's high execution risk (30% failure probability) and uncertain commercialization pathway. While Project B offers higher upside potential, Project A's lower capital requirement, longer visibility period, and alignment with stated diversification objectives provide more reliable value creation for the Board.",
  "quantitative_drivers": [
    "Directionally higher NPV: stable 20% growth over five years vs. 50% growth with 30% failure risk",
    "Lower capital intensity: $50M investment vs. $40M with greater revenue certainty",
    "Longer cash generation visibility: five-year growth runway before stabilization"
  ],
  "qualitative_drivers": [
    "Strategic alignment with company's stated diversification and long-term growth objectives",
    "Quantifiable risk profile (geopolitical, regulatory, competitive) vs. technological execution uncertainty",
    "Manageable execution risk through partnership model despite limited emerging market experience"
  ]
}
```

## Layer A — Rubric-Aligned Checks
|Check|Status|Message|
|-----|------|-------|
|WACC referenced (+2)|PASS|WACC (9%) referenced.|
|Cash referenced (+2)|PASS|Cash ($100 million) referenced.|
|No fabricated NPV/IRR (+2)|PASS|No fabricated figures.|
|Three risks (+1)|PASS|Three risks: ['5.1', '5.2', '5.3']|
|Recommendation present (+1)|PASS|Recommendation present.|
|Allocation sums (+1)|PASS|Sums to 100.0.|
|Consistency [NEW] (+2)|PASS|All sections aligned on: Project A|
|Timing context [NEW] (+1)|PASS|Year 2025 mentioned (partial).|
|NPV/IRR exhibit [NEW] (+1)|PASS|NPV/IRR exhibit referenced (Word table added in render).|
|Upside/downside [NEW] (+1)|PASS|Upside and downside scenarios both present.|
|Explicit comparison [NEW] (+1)|PASS|Explicit NPV and IRR comparison present.|
|Risks anchored [NEW] (+1)|PASS|Risks anchored to Project A (5 mentions vs 0 others).|

## Section Scores
| Section | Conf | Score | Correct | Complete | Clarity | Format | Useful | Review |
|---------|------|-------|---------|----------|---------|--------|--------|--------|
| Executive Summary | 0.74 | 8/10 | 8 | 8 | 9 | 9 | 8 | No |
| 1. Introduction | 0.94 | 9/10 | 9 | 8 | 9 | 9 | 8 | No |
| 2. Project Overview | 0.74 | 9/10 | 9 | 8 | 9 | 9 | 8 | No |
| 3. Financial Analysis | 0.74 | 8/10 | 8 | 9 | 8 | 9 | 8 | No |
| 4. Recommendation | 0.70 | 8/10 | 7 | 8 | 8 | 8 | 7 | No |
| 5. Risk Mitigation | 0.74 | 9/10 | 8 | 9 | 9 | 9 | 8 | No |
| 6. Capital Allocation | 0.67 | 7/10 | 6 | 6 | 8 | 8 | 6 | Yes |
| 7. Organisational Capacity | 0.86 | 9/10 | 9 | 8 | 9 | 9 | 8 | No |
| 8. Conclusion | 0.73 | 8/10 | 7 | 8 | 9 | 9 | 8 | No |

## Summary
- Locked recommendation: Project A
- Flagged: ['capital_allocation']
- Avg confidence: 0.76
- Avg critic score: 8.3/10

## B15 — Download output package

Packages the final deliverables (Board report, dashboard, evaluation report, audit log, verification results, baseline comparison) into a single zip file and downloads it to the local machine. The packaged files form the complete audit trail for this run.

In [ ]:
FINAL_FILES = [
    'board_report.pdf',
    'board_report.docx',
    'dashboard.html',
    'evaluation_report.md',
    'audit_log.json',
    'verification_results.json',
    'baseline_comparison.json',
]

import zipfile
zip_path = OUTPUT_DIR / f'output_{RUN_ID}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in FINAL_FILES:
        fpath = RUN_DIR / fname
        if fpath.exists():
            zf.write(fpath, fname)
            print(f'  Added: {fname}')
        else:
            print(f'  [MISSING]: {fname}')

print(f'Created: {zip_path}')
colab_files.download(str(zip_path))

  Added: board_report.pdf
  Added: board_report.docx
  Added: dashboard.html
  Added: evaluation_report.md
  Added: audit_log.json
  Added: verification_results.json
  Added: baseline_comparison.json
Created: /content/outputs/output_dfedf5.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## B16 — Output verification

A final sanity check that all expected artefacts were produced. Lists each expected file with its size, and prints the canonical facts extracted by Stage 1 so the user can confirm the pipeline ran on the correct document.

In [ ]:
expected = ['board_report.docx','board_report.pdf','dashboard.html',
            'evaluation_report.md','audit_log.json',
            'verification_results.json','baseline_comparison.json']
print(f'Run: {RUN_ID}\n')
all_ok = True
for fname in expected:
    p = RUN_DIR/fname
    if p.exists(): print(f'  OK {fname} ({p.stat().st_size:,} bytes)')
    else: print(f'  MISSING {fname}'); all_ok = False
f = json.loads((RUN_DIR/'facts.json').read_text())
print(f'\nFacts: company={f.get("company_name")} wacc={f.get("wacc")} cash={f.get("available_cash")}')
if all_ok: print('\nAll artefacts present.')


Run: dfedf5

  OK board_report.docx (45,091 bytes)
  OK board_report.pdf (67,632 bytes)
  OK dashboard.html (4,727 bytes)
  OK evaluation_report.md (4,807 bytes)
  OK audit_log.json (69,538 bytes)
  OK verification_results.json (43,089 bytes)
  OK baseline_comparison.json (11,275 bytes)

Facts: company=Tiny-Rod Hit Inc. wacc=9% cash=$100 million

All artefacts present.
